# Task 6 — Music Genre Classification

Compare track-level Librosa tabular features with a log-Mel spectrogram CNN on GTZAN. Both approaches must use the same recording-level split. Do not split spectrogram patches independently.

**Execution note:** run from top to bottom in Google Colab. Save the executed notebook back to this branch so its outputs can be audited.

## 1. Environment and reproducibility

Install project dependencies, record package versions and detect available hardware. Use seed 42 unless the experiment config is deliberately changed.

In [ ]:
# In Colab, clone the repository/checkout this branch, then install:
%pip install -q -r requirements.txt
%pip install -q -e .
import os, platform, random
import numpy as np
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
print('Python:', platform.python_version())
print('NumPy:', np.__version__)


## 2. Dataset acquisition and audit

Download GTZAN from Kaggle into the runtime (do not commit audio). Set `MUSIC_DATA_DIR` to the extracted dataset root. Inspect class counts and audio metadata; flag unreadable files and possible duplicate content before training. Dataset counts must come from the actual downloaded copy.

In [ ]:
from music_genre_classification.data import discover_tracks
tracks = discover_tracks(os.getenv('MUSIC_DATA_DIR', 'data'))
display(tracks.groupby('genre').size().rename('track_count').to_frame())
print('Total recordings:', len(tracks))
display(tracks.head())


## 3. One shared recording-level split

Create and persist one stratified split by original track. All downstream feature rows and spectrograms must map back to these track IDs. Inspect split class counts and verify no ID overlaps.

In [ ]:
from music_genre_classification.data import split_tracks
train_tracks, val_tracks, test_tracks = split_tracks(
    tracks, test_size=0.20, validation_size=0.20, random_seed=SEED
)
for name, part in [('train', train_tracks), ('validation', val_tracks), ('test', test_tracks)]:
    print(name, len(part))
    display(part.groupby('genre').size().rename('count').to_frame())
assert not (set(train_tracks.track_id) & set(val_tracks.track_id))
assert not (set(train_tracks.track_id) & set(test_tracks.track_id))
assert not (set(val_tracks.track_id) & set(test_tracks.track_id))


## 4. Tabular feature extraction and baseline models

Extract one aggregated feature vector per full recording. Cache the result locally. Fit imputation/scaling only within model pipelines on training data. Choose the model using validation macro-F1; evaluate the chosen model once on the held-out test set.

In [ ]:
from music_genre_classification.features import build_feature_table, save_feature_table
from music_genre_classification.models import build_baseline_models
from music_genre_classification.evaluation import feature_matrix, evaluate_classifier
import pandas as pd
feature_cache = 'cache/gtzan_track_features.parquet'
if os.path.exists(feature_cache):
    feature_table = pd.read_parquet(feature_cache)
else:
    feature_table = build_feature_table(tracks, sample_rate=22050, n_mfcc=20)
    save_feature_table(feature_table, feature_cache)
feature_table.head()


In [ ]:
# TODO: join feature rows to the fixed track split by track_id; train each baseline;
# select using validation macro-F1; retain test set untouched until final evaluation.


## 5. Log-Mel spectrogram CNN

Build one fixed-size log-Mel spectrogram per complete track (or fixed track-derived representation). Keep all windows/patches from a recording in that recording's assigned split. Train a compact CNN with early stopping on validation data. Save the preprocessing parameters and model configuration.

In [ ]:
# TODO: implement log-Mel conversion, track-level batching, CNN training and validation.
# TensorFlow is optional and may be installed separately in a Colab GPU runtime.


## 6. Final comparison and error analysis

Evaluate both selected approaches on exactly the same test track IDs. Report accuracy, macro-F1, per-genre precision/recall/F1, confusion matrices, runtime and limitations. Discuss GTZAN's known data-quality and duplication concerns. Do not claim results for experiments that were not run.

In [ ]:
# TODO: consolidate results and generate comparison plots/tables after both approaches run.
